# Population Density by Municipality (1996–2025)

This notebook computes **population density** (inhabitants per km²) for each Spanish
municipality and year, using cleaned population data from the municipal census
(*Padrón Municipal*) and official municipal boundaries from the IGN OGC API.

Population density is a derived demographic attribute at the municipality–year level,
exported as a standalone dataset for integration in subsequent analytical workflows.

> **Update — April 2026:** Municipal boundaries sourced from
> `mun_geographic_administrative_hierarchy.gpkg` (output of notebook 00),
> replacing the previous CNIG shapefile workflow. Series extended to 2025.
> `Mun_Code` extracted directly from GeoPackage — no NATCODE parsing required.

## Definition

Population density measures the number of inhabitants per square kilometre
in a given territorial unit.

$$
\text{Population Density} = \frac{\text{Total Population}}{\text{Area (km}^2\text{)}}
$$

**Sources:**
- Population data: INE — *Padrón Municipal de Habitantes* (table 29005, JSON API)
- Municipal boundaries: IGN OGC API-Features — `api-features.ign.es`
  (via `mun_geographic_administrative_hierarchy.gpkg`, notebook 00)

**CRS methodology:**
Area calculations are performed separately for each territory to minimise
projection distortion:
- Peninsula + Balearic Islands → ETRS89 / UTM zone 30N (**EPSG:25830**)
- Canary Islands → REGCAN95 / UTM zone 28N (**EPSG:4083**)

Canary Islands are identified by province codes `35` (Las Palmas) and `38` (Santa Cruz de Tenerife).

In [ ]:
"""
Notebook: 02_demography_population_density.ipynb
Author:   Juan Zotes
Created:  exploratory phase
Last updated: 2026-04

Purpose:
    Compute population density (inhabitants per km2) per municipality and year.

Inputs:
    - data/demography/processed/01_padron_clean_1996_2025.csv
      (output of notebook 01)
    - data/spatial/derived/mun_geographic_administrative_hierarchy.gpkg
      layer: municipalities_hierarchy
      (output of notebook 00)

Output:
    - data/demography/derived/02_population_density_1996_2025.csv

Spatial unit:
    Municipality (INE 5-digit code, Mun_Code)

Notes:
    - Only 'Total' population rows used (sex-disaggregated rows excluded)
    - Areas computed in projected CRS (EPSG:25830 / EPSG:4083)
    - Area is static (current boundaries applied to full time series)
    - NaN density where population is missing or area is zero
"""

## 1. Load required libraries and configure environment

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import geopandas as gpd

warnings.filterwarnings('ignore')

In [ ]:
# Project root
PROJECT_ROOT = Path(
    r"C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain"
)

# Input paths
PADRON_FILE = (
    PROJECT_ROOT / "data" / "demography" / "processed"
    / "01_padron_clean_1996_2025.csv"
)
GPKG_FILE = (
    PROJECT_ROOT / "data" / "spatial" / "derived"
    / "mun_geographic_administrative_hierarchy.gpkg"
)
GPKG_LAYER = "municipalities_hierarchy"

# Output path
DERIVED_DIR = PROJECT_ROOT / "data" / "demography" / "derived"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = DERIVED_DIR / "02_population_density_1996_2025.csv"

# CRS for area computation (projected, units in metres)
CRS_PENINSULA = "EPSG:25830"   # ETRS89 / UTM zone 30N
CRS_CANARIAS  = "EPSG:4083"    # REGCAN95 / UTM zone 28N

# Province codes for Canary Islands
PROV_CANARIAS = ['35', '38']

# Verify input files exist
for label, path in [("Padron CSV", PADRON_FILE), ("GeoPackage", GPKG_FILE)]:
    status = "OK" if path.exists() else "NOT FOUND"
    print(f"  [{status}]  {label}: {path}")

## 2. Load municipal boundaries and compute areas

Municipal geometries are loaded from the GeoPackage produced in notebook 00.
That file already contains clean `Mun_Code` values (5-digit strings) and
excludes all anomalous entities (plazas de soberanía, mancomunidades, Gibraltar).

Areas are computed after reprojecting each territory to its appropriate UTM zone.

In [ ]:
# Load geometries from GeoPackage (notebook 00 output)
gdf = gpd.read_file(GPKG_FILE, layer=GPKG_LAYER)

# Ensure Mun_Code is a 5-digit string
gdf['Mun_Code'] = gdf['Mun_Code'].astype(str).str.zfill(5)

print(f"GeoPackage loaded: {len(gdf)} municipalities | CRS: {gdf.crs}")
print(f"Columns: {gdf.columns.tolist()}")

In [ ]:
# Split by territory for separate CRS reprojection
mask_canarias = gdf['Prov_Code'].isin(PROV_CANARIAS)

gdf_peninsula = gdf[~mask_canarias].to_crs(CRS_PENINSULA)
gdf_canarias  = gdf[mask_canarias].to_crs(CRS_CANARIAS)

print(f"Peninsula + Baleares: {len(gdf_peninsula)} municipalities → {CRS_PENINSULA}")
print(f"Canarias            : {len(gdf_canarias)} municipalities → {CRS_CANARIAS}")

# Compute area in km² (geometry units are metres after reprojection)
gdf_peninsula['Area_km2'] = gdf_peninsula.geometry.area / 1_000_000
gdf_canarias['Area_km2']  = gdf_canarias.geometry.area  / 1_000_000

# Combine into area lookup table
area_lookup = pd.concat([
    gdf_peninsula[['Mun_Code', 'Area_km2']],
    gdf_canarias[['Mun_Code', 'Area_km2']]
], ignore_index=True)

print(f"\nArea lookup: {len(area_lookup)} municipalities")

# Validate
assert area_lookup['Mun_Code'].nunique() == 8132, (
    f"Expected 8,132 unique municipalities, got {area_lookup['Mun_Code'].nunique()}"
)
assert (area_lookup['Area_km2'] > 0).all(), "All areas must be positive"
print("All areas positive")

print(f"\nArea statistics (km2):")
print(area_lookup['Area_km2'].describe().round(2).to_string())

## 3. Load cleaned demographic data

The cleaned Padrón CSV (output of notebook 01) is loaded and filtered to the
`Total` category — total population, excluding sex-disaggregated rows.

In [ ]:
# Load cleaned Padron (notebook 01 output)
df = pd.read_csv(PADRON_FILE, dtype={"Mun_Code": str})

# Ensure 5-digit format
df['Mun_Code'] = df['Mun_Code'].str.zfill(5)

print(f"Loaded: {len(df):,} records")
print(f"Years : {df['Year'].min()} - {df['Year'].max()}")
print(f"Municipalities: {df['Mun_Code'].nunique():,}")
print(f"Categories: {df['Cat'].unique().tolist()}")

In [ ]:
# Filter to Total population only
df_total = df[df['Cat'] == 'Total'].copy()

print(f"Records after filtering for 'Total': {len(df_total):,}")
print(f"Municipalities: {df_total['Mun_Code'].nunique():,}")
print(f"NaN in Pop    : {df_total['Pop'].isna().sum():,}")
print()
print(df_total.head())

## 4. Merge population data with municipal areas

In [ ]:
# Left join: all Padron rows kept, area added from lookup
df_merged = df_total.merge(
    area_lookup,
    on='Mun_Code',
    how='left',
    validate='m:1'
)

print(f"Merged dataset: {len(df_merged):,} records")

# Check for municipalities without area data
missing_area = df_merged[df_merged['Area_km2'].isna()]['Mun_Code'].nunique()
if missing_area > 0:
    print(f"WARNING: {missing_area} municipalities without area data")
    unmatched = df_merged[df_merged['Area_km2'].isna()][['Mun_Code', 'Mun']].drop_duplicates()
    print(unmatched.to_string(index=False))
else:
    print("All municipalities matched with area data")

print()
print(df_merged.tail())

## 5. Compute population density

Population density = Total Population / Area (km²).

- NaN population → NaN density (propagates naturally)
- Area ≤ 0 → NaN density (safeguard, should not occur after validation above)

In [ ]:
# Compute density — NaN propagates naturally from Pop or Area_km2
df_merged['Pop_Density'] = df_merged['Pop'] / df_merged['Area_km2']

# Safeguard: set density to NaN where area is zero or negative
df_merged.loc[df_merged['Area_km2'] <= 0, 'Pop_Density'] = float('nan')

print("Pop_Density statistics (inhabitants per km2):")
print(df_merged['Pop_Density'].describe().round(2).to_string())

## 6. Quality control

In [ ]:
# Sanity checks
n_negative = (df_merged['Pop_Density'] < 0).sum()
n_infinite = (df_merged['Pop_Density'] == float('inf')).sum()
n_nan      = df_merged['Pop_Density'].isna().sum()

assert n_negative == 0, f"Negative density values found: {n_negative}"
assert n_infinite == 0, f"Infinite density values found: {n_infinite}"

print(f"Negative density values : {n_negative}")
print(f"Infinite density values : {n_infinite}")
print(f"NaN density values      : {n_nan}")
print()
print("All sanity checks passed")

In [ ]:
# Inspect highest and lowest density municipalities (most recent year)
latest_year = df_merged['Year'].max()
df_latest = df_merged[df_merged['Year'] == latest_year]

print(f"=== Highest density ({latest_year}) ===")
print(df_latest.nlargest(10, 'Pop_Density')[
    ['Mun_Code', 'Mun', 'Pop', 'Area_km2', 'Pop_Density']
].to_string(index=False))

print(f"\n=== Lowest density ({latest_year}) ===")
print(df_latest.nsmallest(10, 'Pop_Density')[
    ['Mun_Code', 'Mun', 'Pop', 'Area_km2', 'Pop_Density']
].to_string(index=False))

## 7. Prepare and export final dataset

Output columns:
- `Year` — census year
- `Mun_Code` — 5-digit municipality code (string, leading zeros preserved)
- `Mun` — municipality name
- `Area_km2` — municipal area in km² (static: current boundaries applied to full series)
- `Pop_Density` — inhabitants per km²

In [ ]:
# Select and order final columns
density_df = df_merged[['Year', 'Mun_Code', 'Mun', 'Area_km2', 'Pop_Density']].copy()

# Preserve leading zeros
density_df['Mun_Code'] = density_df['Mun_Code'].astype(str).str.zfill(5)

# Round for readability
density_df['Area_km2']    = density_df['Area_km2'].round(4)
density_df['Pop_Density'] = density_df['Pop_Density'].round(2)

# Sort
density_df = density_df.sort_values(['Mun_Code', 'Year']).reset_index(drop=True)

# Final summary
print("\n" + "="*70)
print("FINAL DATASET SUMMARY")
print("="*70)
print(f"\n  Rows          : {len(density_df):,}")
print(f"  Municipalities: {density_df['Mun_Code'].nunique():,}")
print(f"  Years         : {density_df['Year'].min()} - {density_df['Year'].max()}")
print(f"  NaN density   : {density_df['Pop_Density'].isna().sum():,}")
print()
print(density_df.head(10).to_string(index=False))

# Final assertions
assert density_df['Mun_Code'].nunique() == 8132
assert (density_df['Mun_Code'].str.len() == 5).all()
assert density_df['Year'].max() == 2025
print("\nAll assertions passed. Dataset ready for export.")
print("="*70)

In [ ]:
# Export to CSV
density_df.to_csv(
    OUTPUT_FILE,
    index    = False,
    encoding = "utf-8-sig",   # UTF-8 with BOM — compatible with Excel and QGIS
)

print("\n" + "="*70)
print("EXPORT COMPLETE")
print("="*70)
print(f"\n  File     : {OUTPUT_FILE}")
print(f"  Size     : {OUTPUT_FILE.stat().st_size / 1024 / 1024:.1f} MB")
print(f"  Rows     : {len(density_df):,}")
print(f"  Encoding : UTF-8 with BOM (utf-8-sig)")
print(f"  Sep      : comma")
print("\n  Columns:")
for col in density_df.columns:
    print(f"    {col}")

## Conclusion

Population density (inhabitants per km²) has been computed for all 8,132 Spanish
municipalities across the 1996–2025 time series. Key methodological decisions:

- Municipal boundaries from `mun_geographic_administrative_hierarchy.gpkg` (notebook 00)
- Areas computed in projected CRS (EPSG:25830 for Peninsula + Baleares,
  EPSG:4083 for Canary Islands)
- Area is static: current boundaries applied to all years in the series
- Only `Total` population used (sex disaggregation excluded)

The dataset is exported as `02_population_density_1996_2025.csv` and will be
integrated with other demographic indicators in the geodatabase analítica
municipal (Paper 3, Phase A).

### Next steps

The next notebook (p0) integrates the cleaned Padrón with the Goerlich 2016
typology and produces the consolidated analytical dataset for Paper 1.